In [ ]:
import pandas as pd
import numpy as np

ufo = pd.read_csv("scrubbed.csv", low_memory=False)

print("First 10 rows:")
print(ufo.head(10))

print("\nLast 5 rows:")
print(ufo.tail())

print("\nDataset dimensions:")
print(ufo.shape)

print("\nColumn names:")
print(ufo.columns.tolist())

print("\nData types:")
print(ufo.dtypes)

print("\nMissing values:")
print(ufo.isnull().sum())

print("\nUnique countries:")
print(ufo["country"].nunique())

print("\nUnique shapes:")
print(ufo["shape"].nunique())

In [ ]:
selected_columns = ufo[
    [
        "datetime",
        "city",
        "country",
        "shape",
        "duration (seconds)"
    ]
]

print(selected_columns.head())

us_reports = ufo[ufo["country"] == "us"]
print("US reports:", len(us_reports))

triangle_reports = ufo[
    ufo["shape"].str.lower() == "triangle"
]
print(triangle_reports.head())

us_triangles = ufo[
    (ufo["country"] == "us") &
    (ufo["shape"].str.lower() == "triangle")
]
print(us_triangles.head())

ufo["duration (seconds)"] = pd.to_numeric(
    ufo["duration (seconds)"],
    errors="coerce"
)

long_reports = ufo[
    ufo["duration (seconds)"] > 600
]
print(long_reports.head())

longest_reports = ufo.dropna(
    subset=["duration (seconds)"]
).nlargest(10, "duration (seconds)")

print(
    longest_reports[
        ["datetime", "city", "shape", "duration (seconds)"]
    ]
)

In [ ]:
ufo_clean = pd.read_csv("scrubbed.csv", low_memory=False)

rows_before = len(ufo_clean)

ufo_clean = ufo_clean.drop_duplicates().copy()

rows_after = len(ufo_clean)

print("Duplicates removed:", rows_before - rows_after)

for column in ["city", "country", "shape"]:
    ufo_clean[column] = (
        ufo_clean[column]
        .str.strip()
    )

ufo_clean["city"] = ufo_clean["city"].str.title()
ufo_clean["country"] = ufo_clean["country"].str.lower()
ufo_clean["shape"] = ufo_clean["shape"].str.title()

ufo_clean["datetime"] = pd.to_datetime(
    ufo_clean["datetime"],
    errors="coerce"
)

ufo_clean["duration (seconds)"] = pd.to_numeric(
    ufo_clean["duration (seconds)"],
    errors="coerce"
)

print(ufo_clean.isnull().sum())

print(
    ufo_clean[
        ["country", "shape", "duration (seconds)"]
    ].isnull().sum()
)

In [ ]:
# 10 most common shapes
top_shapes = ufo_clean["shape"].value_counts().head(10)
print("Most common shapes:")
print(top_shapes)

# Reports by country
country_counts = ufo_clean["country"].value_counts()
print("\nReports by country:")
print(country_counts.head(10))

# Five countries with the most reports
print("\nTop five countries:")
print(country_counts.head(5))

# Average and median duration
valid_durations = ufo_clean[
    ufo_clean["duration (seconds)"].notna()
    & (ufo_clean["duration (seconds)"] >= 0)
]

print("\nAverage duration in seconds:")
print(valid_durations["duration (seconds)"].mean())

print("\nMedian duration in seconds:")
print(valid_durations["duration (seconds)"].median())

# Average duration by shape
average_duration_by_shape = (
    valid_durations
    .groupby("shape")["duration (seconds)"]
    .mean()
    .sort_values(ascending=False)
)

print("\nAverage duration by shape:")
print(average_duration_by_shape.head(10))

# Number of reports per year
ufo_clean["year"] = ufo_clean["datetime"].dt.year

reports_by_year = (
    ufo_clean["year"]
    .value_counts()
    .sort_index()
)

print("\nReports by year:")
print(reports_by_year)

print("\nYear with the most reports:")
print(reports_by_year.idxmax())

print("Number of reports in that year:")
print(reports_by_year.max())

# Reports per calendar month, across all years
ufo_clean["month"] = ufo_clean["datetime"].dt.month

reports_by_month = (
    ufo_clean["month"]
    .value_counts()
    .sort_index()
)

print("\nReports by month number:")
print(reports_by_month)

In [ ]:
# Create duration in minutes
ufo_clean["duration_minutes"] = (
    ufo_clean["duration (seconds)"] / 60
)

# Create a year column
# Already created above; we can inspect it:
print(ufo_clean[["datetime", "year"]].head())

# Reports lasting at least one hour
one_hour_reports = ufo_clean[
    ufo_clean["duration (seconds)"] >= 3600
]

print("\nReports lasting at least one hour:")
print(
    one_hour_reports[
        ["datetime", "city", "country", "shape",
         "duration (seconds)", "duration_minutes"]
    ].head(20)
)

print("Total reports lasting at least one hour:", len(one_hour_reports))

# Average duration by shape, including only shapes
# with at least 20 valid duration records
shape_duration_stats = (
    valid_durations
    .groupby("shape")["duration (seconds)"]
    .agg(["count", "mean", "median"])
)

qualified_shapes = shape_duration_stats[
    shape_duration_stats["count"] >= 20
].sort_values("mean", ascending=False)

print("\nTop five shapes by average duration")
print(qualified_shapes.head(5))